
## Tags: **PRP** pronoun, **MD** modal, **VB** verb, **DT** determiner, **NN** noun.

#### What we will do

| Phase | Step | What happens |
| --- | --- | --- |
| **The model** | 1. Vocabulary | Split the text on spaces and keep each new word |
| | 2. Table A | P(tag \| previous tag), with the `<s>` row as π |
| | 3. Table B | P(word \| tag) |
| **Viterbi** | 4. Initialization | First column: π(tag) × P(word \| tag) |
| | 5. Recursion | Each cell: max of previous × A × B, and save a backpointer |
| | 6. Termination | Take the biggest cell in the last column |
| | 7. Backtrace | Follow the backpointers to get the best tags |
| **Testing** | 8–9 | Tag new sentences, and see what happens with an unknown word |

In [1]:
corpus = [
    "I can book the flight",
    "you read the book",
    "the man can read",
    "I book a room",
    "the can fell",
    "they will read a book",
    "the book fell",
]

In [2]:
import re
from collections import Counter

In [8]:
def build_vocab(corpus):
    vocab = []
    for sentence in corpus:
        sentence = re.sub(r'[^\w\s]', '', sentence.lower())
        words = sentence.split()
        for word in words:
            if word not in vocab:
                vocab.append(word)
    return vocab
vocab = build_vocab(corpus)
print("Vocabulary:", vocab,"\n","len = ",len(vocab)," word" )

Vocabulary: ['i', 'can', 'book', 'the', 'flight', 'you', 'read', 'man', 'a', 'room', 'fell', 'they', 'will'] 
 len =  13  word


In [29]:
def tokenize_sentence(corpus):
    vocab = []
    vocab= [re.sub(r'[^\w\s]', '', sentence.lower()).split() for sentence in corpus if sentence not in vocab]
    return vocab

sentence_list = tokenize_sentence(corpus)
for sentence in sentence_list:
    print(sentence)

['i', 'can', 'book', 'the', 'flight']
['you', 'read', 'the', 'book']
['the', 'man', 'can', 'read']
['i', 'book', 'a', 'room']
['the', 'can', 'fell']
['they', 'will', 'read', 'a', 'book']
['the', 'book', 'fell']


In [58]:
states = ["PRP", "MD", "VB", "DT", "NN"]

# A[previous][next] = P(next | previous). Missing pairs = 0.
A = {
    "<s>": {"PRP": 4/7, "DT": 3/7},          # this row is pi
    "PRP": {"MD": 2/4, "VB": 2/4},
    "MD":  {"VB": 3/3},
    "VB":  {"DT": 4/7, "</s>": 3/7},
    "DT":  {"NN": 7/7},
    "NN":  {"MD": 1/7, "VB": 2/7, "</s>": 4/7},
}
 
# B[tag][word] = P(word | tag). Missing words = 0.
B = {
    "PRP": {"i": 2/4, "you": 1/4, "they": 1/4},
    "MD":  {"can": 2/3, "will": 1/3},
    "VB":  {"book": 2/7, "read": 3/7, "fell": 2/7},
    "DT":  {"the": 5/7, "a": 2/7},
    "NN":  {"can": 1/7, "book": 3/7, "flight": 1/7, "man": 1/7, "room": 1/7},
}

#A and B is a dictionary inside a dictionary, so you need two steps to reach one cell, just like a row and a column:
def a(prev, nxt):
    if prev not in A and nxt not in A[prev]:
        return 0
    return A.get(prev, {}).get(nxt, 0)
 
def b(tag, word):
    if tag not in B and word not in B[tag]:
        return 0
    return B.get(tag, {}).get(word, 0)

def print_table(title, table, rows, cols):
    print(title)
    print(f"{'':>6}" + "".join(f"{c:>8}" for c in cols))      # header row
    for r in rows:
        line = f"{r:>6}"                                       # row name
        for c in cols:
            p = table.get(r, {}).get(c, 0)                     # missing = 0
            line += f"{p:>8.2f}" if p > 0 else f"{'0':>8}"
        print(line)
    print()


print_table("Table A: P(next | previous)", A, ["<s>"] + states, states + ["</s>"])
print_table("Table B: P(word | tag)", B, states, vocab)
 

Table A: P(next | previous)
           PRP      MD      VB      DT      NN    </s>
   <s>    0.57       0       0    0.43       0       0
   PRP       0    0.50    0.50       0       0       0
    MD       0       0    1.00       0       0       0
    VB       0       0       0    0.57       0    0.43
    DT       0       0       0       0    1.00       0
    NN       0    0.14    0.29       0       0    0.57

Table B: P(word | tag)
             i     can    book     the  flight     you    read     man       a    room    fell    they    will
   PRP    0.50       0       0       0       0    0.25       0       0       0       0       0    0.25       0
    MD       0    0.67       0       0       0       0       0       0       0       0       0       0    0.33
    VB       0       0    0.29       0       0       0    0.43       0       0       0    0.29       0       0
    DT       0       0       0    0.71       0       0       0       0    0.29       0       0       0       0
    NN  

In [26]:
def print_matrix(name, matrix):
    print(name, "=")
    for row in matrix:
        print(row)
    print()


### Q: What is `viterbi[prev][w-1]`, and why do we need it?

**What:** the score of the best path from the first word up to the **previous word**, ending in tag `prev`.

**Why:** a word's tag depends on its **context**. The same word *book* is a verb in *I **can book** a flight* and a noun in *I read **the book***. Table B gives the same numbers in both, so only **what came before** can decide.

Like a trip to Baghdad: to know the best route to a town, you need **the total distance so far + the last road**:

    new score = viterbi[prev][w-1]  ×  a × b
                 (the path so far)    (one more step)

**In one line:** without the past, each word guesses alone; with the past, each word is tagged in context.

In [82]:
def viterbi_tag(obs,states):
    num_words = len(obs)                  # T: number of columns
    num_tags = len(states)                # N: number of rows

    # 1----------------------------------------------------------------------#
    # create a path probability matrix viterbi[N, T] and a backpointer matrix
    # create the lattice viterbi[tag][word] and the backpointer table (all empty)
    viterbi = [[0.0] * num_words for _ in range(num_tags)]
    backpointer = [[None] * num_words for _ in range(num_tags)]

    #print_matrix("viterbi", viterbi)
    #print_matrix("backpointer", backpointer)
 
    # initialization step: first word, viterbi[t][0] = pi(t) * b(t, first word)
    for t in range(num_tags):
        viterbi[t][0] = a("<s>", states[t]) * b(states[t], obs[0])
        backpointer[t][0] = None
 
    
    #print_matrix("viterbi", viterbi)
    #print_matrix("backpointer", backpointer)

    # range(N) loops over the tags (s, s2).
    # range(T) loops over the words (t).
    # N = number of states (tags) = rows == s , s2.
    # T = number of time steps (words) = columns == t.
    # 3 -------------------------------------------------------------------#
    # recursion step: viterbi[t][w] = max over prev of viterbi[prev][w-1] * a(prev, t) * b(t, word)
       # recursion step: viterbi[t][w] = max over prev of viterbi[prev][w-1] * a(prev, t) * b(t, word)
    for w in range(1, num_words):
        print(f"\n===== column {w}: word '{obs[w]}' (previous word: '{obs[w-1]}') =====")
        for t in range(num_tags):
            scores = []
            for prev in range(num_tags):
                p = viterbi[prev][w-1] * a(states[prev], states[t]) * b(states[t], obs[w])
                scores.append(p)

            best = max(scores)
            viterbi[t][w] = best
            backpointer[t][w] = scores.index(best) if best > 0 else None

            print(f"  '{obs[w]}' as {states[t]:<4}: scores = {[round(x, 4) for x in scores]}  → best = {best:.4f}")

        print(f"  viterbi column for '{obs[w]}'    :", [round(viterbi[t][w], 4) for t in range(num_tags)])
        print(f"  backpointer column for '{obs[w]}':", [states[backpointer[t][w]] if backpointer[t][w] is not None else '-' for t in range(num_tags)])
    # 4 -------------------------------------------------------------------#

# termination step: the biggest cell in the last column
    last = num_words - 1
    scores = []
    for t in range(num_tags):
        scores.append(viterbi[t][last])

    print(f"\n===== termination: last word '{obs[last]}' =====")
    print("  last column:", [f"{states[t]}={round(scores[t], 6)}" for t in range(num_tags)])

    best = max(scores)
    if best == 0:
        print("  all cells are 0 → no path")
        return None, 0.0
    best_last = scores.index(best)
    best_path_prob = best
    print(f"  biggest cell: '{obs[last]}' as {states[best_last]} = {best:.6f}  → start the backtrace here")

    # 5 -------------------------------------------------------------------#
    # backtrack to find the best path
    # backtrace: follow the backpointers from the last word back to the first
  
    # backtrace: follow the backpointers from the last word back to the first
    print("\n===== backtrace =====")
    path = [best_last]
    print(f"  start: path = {[states[i] for i in path]}")
    for w in range(last, 0, -1):
        prev = backpointer[path[-1]][w]
        print(f"  '{obs[w]}' as {states[path[-1]]} → arrow points to '{obs[w-1]}' as {states[prev]}")
        path.append(prev)
        print(f"         path = {[states[i] for i in path]}")

    path.reverse()
    print(f"  after reverse: path = {[states[i] for i in path]}")
 
    return [(obs[w], states[path[w]]) for w in range(num_words)], best_path_prob
 


In [ ]:

for obs in tokenize_sentence(corpus=corpus):
    tags, prob = viterbi_tag(obs,states=states)
    print("\n\n===== final track =====")

    if tags is None:
        print(f"{' '.join(obs)}: no path (probability 0)")
    else:
        tagged = ' '.join(f"{w}/{t}" for w, t in tags)
        #P is the probability of the best path
        print(f"{tagged}   P = {prob:.3e}")
    break


===== column 1: word 'can' (previous word: 'i') =====
  'can' as PRP : scores = [0.0, 0, 0, 0.0, 0]  → best = 0.0000
  'can' as MD  : scores = [0.0952, 0.0, 0.0, 0.0, 0.0]  → best = 0.0952
  'can' as VB  : scores = [0.0, 0.0, 0, 0.0, 0.0]  → best = 0.0000
  'can' as DT  : scores = [0.0, 0, 0.0, 0.0, 0]  → best = 0.0000
  'can' as NN  : scores = [0.0, 0.0, 0.0, 0.0, 0.0]  → best = 0.0000
  viterbi column for 'can'    : [0.0, 0.0952, 0.0, 0.0, 0.0]
  backpointer column for 'can': ['-', 'PRP', '-', '-', '-']

===== column 2: word 'book' (previous word: 'can') =====
  'book' as PRP : scores = [0.0, 0.0, 0.0, 0.0, 0.0]  → best = 0.0000
  'book' as MD  : scores = [0.0, 0.0, 0.0, 0.0, 0.0]  → best = 0.0000
  'book' as VB  : scores = [0.0, 0.0272, 0.0, 0.0, 0.0]  → best = 0.0272
  'book' as DT  : scores = [0.0, 0.0, 0.0, 0.0, 0.0]  → best = 0.0000
  'book' as NN  : scores = [0.0, 0.0, 0.0, 0.0, 0.0]  → best = 0.0000
  viterbi column for 'book'    : [0.0, 0.0, 0.0272, 0.0, 0.0]
  backpointer c